<!-- NOTEBOOK_METADATA source: "⚠️ Jupyter Notebook" title: "Observability for AgentsKit with Langfuse" sidebarTitle: "AgentsKit" logo: "/images/integrations/agentskit_icon.svg" description: "Trace AgentsKit agent runs, model calls, tools, token usage, and errors in Langfuse with the native observer integration." category: "Integrations" -->

# Integrate Langfuse with AgentsKit

This notebook shows how to send **AgentsKit** agent runs to **Langfuse** for tracing and debugging. The native observer records planning steps, model generations, tool calls, token usage, latency, and errors without changing agent logic.

> **What is AgentsKit?** [AgentsKit](https://www.agentskit.io/docs) is an open-source TypeScript toolkit for composing provider-independent agents from small runtime, tool, memory, retrieval, and observability packages.

> **What is Langfuse?** [Langfuse](https://langfuse.com) is an open-source LLM engineering platform that helps teams trace, debug, and evaluate their LLM applications.

<!-- STEPS_START -->
## Step 1: Install dependencies

Install the runtime, provider adapters, observability package, and Langfuse SDK:

```bash
npm install @agentskit/runtime @agentskit/adapters @agentskit/observability langfuse
```

> This cookbook runs TypeScript with Deno. In a Node.js application, use the same packages with standard npm imports.

## Step 2: Configure the environment

Get Langfuse keys from your project settings in [Langfuse Cloud](https://langfuse.com/cloud) or use a [self-hosted Langfuse](https://langfuse.com/self-hosting) URL. This example uses OpenRouter's free model router; AgentsKit adapters can swap providers without changing the observer.

In [ ]:
Deno.env.set("LANGFUSE_PUBLIC_KEY", "pk-lf-...");
Deno.env.set("LANGFUSE_SECRET_KEY", "sk-lf-...");
Deno.env.set("LANGFUSE_HOST", "https://cloud.langfuse.com"); // 🇪🇺 EU region
// Other regions: 🇺🇸 https://us.cloud.langfuse.com, 🇯🇵 https://jp.cloud.langfuse.com, ⚕️ https://hipaa.cloud.langfuse.com

Deno.env.set("OPENROUTER_API_KEY", "sk-or-v1-...");

## Step 3: Attach the Langfuse observer

AgentsKit's native Langfuse observer maps runtime events to one trace per run, with nested generation and tool spans. Add it to the runtime's `observers` array; no manual span management is required.

In [ ]:
import { openrouter } from "npm:@agentskit/adapters";
import { langfuse } from "npm:@agentskit/observability/langfuse";
import { createRuntime } from "npm:@agentskit/runtime";
import "npm:langfuse";

const runtime = createRuntime({
  adapter: openrouter({
    apiKey: Deno.env.get("OPENROUTER_API_KEY")!,
    model: "openrouter/free",
  }),
  observers: [
    langfuse({
      sessionId: "agentskit-langfuse-demo",
      tags: ["agentskit", "cookbook"],
      flushAt: 1,
    }),
  ],
  systemPrompt: "Answer concisely and explain your reasoning.",
});

## Step 4: Run the agent

Run the agent normally. The observer records the model generation, usage, latency, and final result in Langfuse.

In [ ]:
const result = await runtime.run(
  "Give three practical reasons to add tracing before an AI agent reaches production.",
);

console.log(result.content);
await new Promise((resolve) => setTimeout(resolve, 2000));

## Step 5: View traces in Langfuse

Open [Langfuse Cloud](https://langfuse.com/cloud) to inspect the run, including the prompt, completion, token usage, latency, and error state.

![AgentsKit trace in Langfuse](https://langfuse.com/images/cookbook/integration-agentskit/agentskit-example-trace.jpg)

[Open the public example trace in Langfuse](https://us.cloud.langfuse.com/project/cmrwcz40n01tlad0i1h13cuwy/traces/99a57d08-c581-4be8-9dde-31844e78506c)

<!-- STEPS_END -->

<!-- MARKDOWN_COMPONENT name: "LearnMore" path: "@/components-mdx/integration-learn-more-js.mdx" -->